# **Movie Recommendation System with Explainability**
### Michalis Chazaridis, Rafał Głodek, Igor Kociszewski, Antoni Krzak
---

## The MovieLens 100k Dataset

*   **The Origin:** Created by GroupLens Research at the University of Minnesota. 
*   **The Scope:** It contains roughly 100,000 ratings (specifically ~100,836) applied to ~9,742 movies by 610 unique users.
*   **The Feedback:** The target variable is explicit feedback—users actively rated movies on a scale from 0.5 to 5.0 stars.
*   **The Metadata:** Alongside the ratings, it includes predefined **Genres** and user-generated **Tags** (e.g., "cyberpunk", "mind-bending"). 
---

## Exploratory Data Analysis

[Exploratory Data Analysis](../notebooks/eda.ipynb)

---

## The Dual-Engine Strategy (Hybrid Architecture)

To build a robust recommendation system, we combine two distinct pipelines. This hybrid approach gets the best of both worlds, allowing each engine to cover the other's weaknesses.

*   **Pipeline A: Neural Collaborative Filtering**
    *   **Mechanism:** Uses PyTorch deep learning to map complex, hidden patterns in historical user interactions (who watched what).
    *   **Strength:** Cccurate personalization based on actual human behavior.
    *   **Weakness:** Suffers from the "Cold Start" problem—it struggles to recommend brand-new movies that nobody has rated yet.

*   **Pipeline B: Content-Based Engine**
    *   **Mechanism:** Analyzes the movies (titles, genres, user-generated tags) using NLP techniques like TF-IDF and Cosine Similarity.
    *   **Strength:** Immune to the Cold Start problem. It can recommend a movie the second it is added to the database just by reading its tags.
    *   **Weakness:** Less personalized; tends to recommend a narrow band of movies that are mathematically similar to each other.

*   **Our Goals: Hybrid Model**
    *   **Candidate Generation:** Pipeline B acts as a rapid filter, instantly scanning the database to find candidates.
    *   **Personalized Ranking:** Pipeline A then takes those candidates and evaluates them against the specific user's neural profile.
---

## Techniques Implemented

### 1. Neural Matrix Factorization (NeuMF) Architecture
A custom architecture that learns user preferences from multiple mathematical angles:
*   **Generalized Matrix Factorization (GMF):** It calculates the direct, hidden overlap between a user's latent profile and a movie's latent profile using dot products.
*   **Multi-Layer Perceptron (MLP):** A dense neural network with dropout and batch normalization that learns complex, non-linear behavioral patterns.
*   **Feature Fusion (Wide & Deep):** The final layer concatenates the GMF vector, the MLP vector, and the **explicit movie genres**. This allows the model to base its final star rating on both hidden behavioral math **and** human-readable tags.

### 2. "Cold Start" Mitigation
Standard neural embeddings crash when asked to process a user or movie ID they did not see during the training phase.
*   **Robust ID Mapping:** We implemented a custom `safe_transform` pipeline that explicitly reserves index `0` as an "Unknown (UNK)" token. If a brand-new user or movie enters the system, the architecture defaults to this baseline token rather than throwing an out-of-bounds error.


### 3. Cosine Similarity & TF-IDF
To power the Content Engine (Pipeline B), we rely on Natural Language Processing techniques:
*   **Vectorization:** We use Term Frequency-Inverse Document Frequency (TF-IDF) to convert textual metadata (genres, titles, and tags) into high-dimensional numerical vectors.
*   **Cosine Similarity:** To find matching films, we calculate the Cosine Similarity between these vectors
---

## SHAP Analysis

[Shap Analysis](../shap_analysis.ipynb)

---

## Current Pipeline Results

![Wyniki](resultspic.png)

## Future Work

### 1. End-to-End Hybrid Architecture
Currently, the system operates in two distinct phases (TF-IDF candidate generation followed by Neural ranking). The next step is to create an end-to-end deep learning model that ingests both collaborative behavioral embeddings and content-based metadata simultaneously during the training phase.

### 2. GUI & Live Explainability
Upgrade the Streamlit prototype to a functional web application. Crucially, the goal is to integrate the SHAP DeepExplainer directly into the live user interface, generating interactive explanations alongside the recommendations.

### 3. Tuning & Transformer Embeddings
Move beyond manual tuning by implementing an automated hyperparameter optimization framework (like Optuna) to systematically discover the optimal parameters. Additionally, we aim to replace the TF-IDF vectorizer with a Transformer-based language model (like BERT) to extract deeper semantic meaning from the movie tags and titles.

### 4. Data Scaling
While the MovieLens 100k dataset was ideal for rapid prototyping and architecture design we consider scaling the training pipeline to the the bigger dataset (a few million). This might allow the neural network to uncover much richer, highly complex behavioral patterns.